# Memorization: recurrent vs feedforward delays

Notebook equivalent of `experiments/compare_mem_snn_rsnn.py`, with the defaults from
`configs/perf_MEM.py`. Compare **axonal, synaptic and hybrid** delays on each pathway.
All six models use the same generated samples, labels and batch order, with matched
initial connection weights. Accuracy is measured on the training set only.

On Kaggle, enable a GPU accelerator and Internet for the first setup, then run all cells.
The setup fetches only the library source from the `toi_task` branch of your GitHub fork; an existing checkout
(or a checkout attached as input) can be used by setting `REPO_DIR`.
The selected revision must contain the six comparison networks and `spike_memorization.py`.

Edit the **Configuration** cell for each experiment, then rerun that cell and the
**Run comparison** and **Plot comparison** cells. Each run gets a new output directory;
there is no need to clone again. Training and matching helpers are also editable here.
No experiment script or Python configuration file is imported.

In [4]:
from pathlib import Path
import os
import subprocess
import sys

ON_KAGGLE = Path('/kaggle/working').is_dir()
WORK_DIR = Path('/kaggle/working') if ON_KAGGLE else Path.cwd()
REPO_URL = 'https://github.com/AlbatorZ/DelRec.git'
REPO_REF = 'toi_task'  # This branch contains the MEM comparison networks.
# For a Kaggle input checkout, replace with its /kaggle/input/... path.
REPO_DIR = WORK_DIR / 'DelRec'
INSTALL_DEPENDENCIES = ON_KAGGLE

# Reuse this repository when running locally from its root or notebooks/ directory.
if not ON_KAGGLE:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'src/delrec/networks.py').is_file():
            REPO_DIR = candidate
            break

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'sparse-checkout', 'set', 'src'], check=True)
if not (REPO_DIR / 'src/delrec/datasets/spike_memorization.py').is_file():
    raise FileNotFoundError(f'Set REPO_DIR to a checkout with the MEM library: {REPO_DIR}')

if INSTALL_DEPENDENCIES:
    # Keep Kaggle's installed CUDA-enabled PyTorch. Install only the extras used here.
    # SpikingJelly is pinned to the same source revision as requirements.txt.
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'DCLS==0.1.1', 'numpy', 'matplotlib', 'prettytable',
                    'spikingjelly @ git+https://github.com/fangwei123456/spikingjelly.git@d4fee3a1715bf42ce15f52fde1b1d4709d7ea25e'],
                   check=True)

sys.path.insert(0, str(REPO_DIR / 'src'))
os.environ.setdefault('MPLCONFIGDIR', str(WORK_DIR / '.matplotlib'))
print(f'Library: {REPO_DIR / "src"}')

Library: /Users/mattia/Documents/CentraleSupelec/3A/filiereRecherche/EtudeDeCas/Code/DelRec/src


In [5]:
from copy import deepcopy
from datetime import datetime
import csv
import json

import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader
from spikingjelly.activation_based import neuron, surrogate

from delrec import networks
from delrec.datasets.spike_memorization import SpikeMemorization
from delrec.delay_layers import axonal_recdel
from delrec.networks import dcls_module, learned_delay_parameter
from delrec.utils import reset_states, seed_everything

print(f'PyTorch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}')

PyTorch: 2.5.1; CUDA available: False


## Configuration

These are the current `perf_MEM.py` defaults (256 samples, 16 classes, 200 epochs).
`model` is selected automatically for each of the six runs. Keep `dataset_seed` fixed
when comparing settings. Change values **inside the class** so derived padding and
position bounds are recalculated when this cell runs. For a quick check, use
`epochs = 2`, `num_samples = 16`, and `hidden_layers = [8]`.

`DEVICE = "auto"` uses CUDA if available, otherwise CPU. Results go to a fresh folder
under `OUTPUT_ROOT`. The setup retains the environment's PyTorch version; this is not
a recreation of every dependency pin in the repository's reference environment.

In [6]:
class Config:
    dataset = "MEM"
    model = "SNN_recurrent_and_feedforward_delays"
    seed = 0
    dataset_seed = 0
    task_type = "temporal"
    num_samples = 256
    input_size = 16
    time_window = 32
    output_size = 16
    input_gain = 1.0
    hidden_layers = [64]
    epochs = 200
    batch_size = 64
    num_workers = 0
    cpu_threads = 1
    readout = "mean"  # mean, sum, or last temporal output

    bias = False
    use_batch_norm = False
    feedforward_dropout_rate = 0.0
    recurrent_dropout_rate = 0.0
    init_ff_weights = "default"
    init_dcls_weights = "default"
    no_delay_in_first_layer = False
    no_delay_in_last_layer = False
    no_recurrence_in_last_layer = False

    neuron_module = neuron.LIFNode
    surrogate_function = surrogate.ATan(alpha=2.0)
    tau = 2.0
    decay_input = False
    v_threshold = 1.0
    v_reset = 0.0
    detach_reset = False
    step_mode = "m"
    backend = "torch"
    store_v_seq = False

    init_rec_weights = "orthogonal"
    rec_delay_init_gain = 0.5
    use_rec_bias = True
    init_rec_delay = "uniform"
    init_recdel_offset = 0.0
    max_rec_delay = 8.0
    delay_std_init = 2.0
    use_sig_p = False
    sigma_init = 0.0
    sigma_decay = 0.95

    #Hybrid delays configuration
    hybrid_max_synaptic_delay = 4  # fixed integer offsets in [0, 4] on both pathways
    hybrid_delay_seed = 123  # independent of dataset and model seed
    round_delays = False
    round_pos_each_epoch = False

    DCLSversion = "gauss"
    kernel_count = 1
    max_feedforward_delay = 9
    left_padding = max_feedforward_delay - 1
    right_padding = 0  # causal, length-preserving convolution
    init_pos_a = -(max_feedforward_delay // 2)
    init_pos_b = max_feedforward_delay // 2
    siginit = 1.0  # scheduled to 0.23 in the first half of training

    lr_w = 0.005
    lr_positions = 0.08
    weight_decay = 0.0
    grad_clip = 1.0


config = Config()
DEVICE = 'auto'  # auto, cuda, or cpu
OUTPUT_ROOT = WORK_DIR / 'exp' / 'MEM' / 'delay_location_comparison'

## Training helpers

AdamW uses separate learning rates for weights and delays, with cosine scheduling.
DCLS widths and recurrent smoothing follow the original schedules. Every epoch ends
with a measurement on the entire training set, retaining fractional delays.
These helpers are copied into the notebook so they can be edited independently.

In [7]:
def get_dcls_sigma_for_epoch(config, epoch: int):

    if getattr(config, "DCLSversion", None) != "gauss":
        return 0.23

    total_epochs = max(1, config.epochs)

    decay_horizon = max(1, total_epochs // 2)

    sigma_min = 0.23
    if epoch >= decay_horizon:
        return sigma_min

    if config.siginit <= sigma_min:
        return sigma_min

    alpha = (sigma_min / float(config.siginit)) ** (1.0 / decay_horizon)
    sigma = float(config.siginit) * (alpha ** epoch)
    return max(sigma, sigma_min)


def make_optimizer(model, config):
    positions = []
    for module in model.modules():
        if isinstance(module, axonal_recdel):
            positions.append(learned_delay_parameter(module, 'recurrent_delays'))
            if hasattr(module, "p_spread"):
                positions.append(module.p_spread)
        elif isinstance(module, dcls_module):
            positions.append(learned_delay_parameter(module, 'P'))
            # Width is scheduled explicitly, rather than optimized.
            if config.DCLSversion == "gauss":
                module.SIG.requires_grad_(False)
    position_ids = {id(p) for p in positions}
    weights = [p for p in model.parameters() if p.requires_grad and id(p) not in position_ids]
    return torch.optim.AdamW([
        {"params": weights, "lr": config.lr_w, "weight_decay": config.weight_decay},
        {"params": [p for p in positions if p.requires_grad],
         "lr": config.lr_positions, "weight_decay": 0.0},
    ])


def set_epoch(model, config, epoch):
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, axonal_recdel):
                module.update_sigma(epoch)
            elif isinstance(module, dcls_module) and config.DCLSversion == "gauss":
                module.SIG.fill_(get_dcls_sigma_for_epoch(config, epoch))


def run_epoch(loader, model, device, config, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for inputs, labels in loader:
            inputs = inputs.permute(1, 0, 2).contiguous().to(device)
            labels = labels.to(device)
            reset_states(model)
            if training:
                optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            if config.readout == "mean":
                logits = outputs.mean(0)
            elif config.readout == "sum":
                logits = outputs.sum(0)
            elif config.readout == "last":
                logits = outputs[-1]
            else:
                raise ValueError(f"Unknown readout: {config.readout}")
            loss = F.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite memorization loss")
            if training:
                loss.backward()
                if config.grad_clip > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                optimizer.step()
                if hasattr(model, "clamp_delays"):
                    model.clamp_delays()
            total_loss += loss.item() * labels.numel()
            correct += (logits.argmax(1) == labels).sum().item()
            count += labels.numel()
    reset_states(model)
    return {"loss": total_loss / count, "accuracy_percent": 100.0 * correct / count,
            "correct": correct, "num_samples": count}

In [8]:
def plot_results(history, final, config, run_dir):
    fig, (loss_ax, acc_ax) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
    loss_ax.plot([r["epoch"] for r in history], [r["loss"] for r in history])
    loss_ax.set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Training set (after each epoch)")
    loss_ax.grid(alpha=0.25)
    accuracy = final["accuracy_percent"]
    acc_ax.bar(["Final training accuracy"], [accuracy], width=0.45)
    acc_ax.set(ylabel="Accuracy (%)", ylim=(0, 110))
    acc_ax.axhline(100 / config.output_size, color="gray", linestyle="--", label="Uniform random guess")
    acc_ax.text(0, accuracy + 2, f"{accuracy:.2f}% ({final['correct']}/{final['num_samples']})", ha="center")
    acc_ax.legend(loc="lower right")
    fig.suptitle(f"{config.model}\n{config.task_type}, {config.num_samples} samples, seed {config.seed}")
    fig.savefig(run_dir / "training_summary.png", dpi=180)
    fig.savefig(run_dir / "training_summary.pdf")
    plt.close(fig)


def run(config, device, out, model):
    """Run one complete experiment, optionally with a preinitialized model."""
    torch.set_num_threads(config.cpu_threads)
    seed_everything(config.seed, is_cuda=torch.cuda.is_available())
    dataset = SpikeMemorization(config.num_samples, config.input_size, config.time_window,
                               config.output_size, config.dataset_seed, config.input_gain, config.task_type)
    loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(config.seed), num_workers=config.num_workers)
    # Same dataset, deterministic order: this is training-set measurement, not a split.
    measure_loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=False,
                                num_workers=config.num_workers)
    model = model.to(device)
    optimizer = make_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    run_dir = Path(out)
    run_dir.mkdir(parents=True, exist_ok=True)
    if (run_dir / "config.json").exists():
        raise FileExistsError(f"Run already exists: {run_dir}")
    settings = {k: getattr(config, k) for k in dir(config)
                if not k.startswith("_") and not callable(getattr(config, k))}
    settings["device"] = str(device)
    settings["surrogate_function"] = repr(config.surrogate_function)
    (run_dir / "config.json").write_text(json.dumps(settings, indent=2))
    torch.save({"inputs": dataset.inputs, "labels": dataset.labels}, run_dir / "dataset.pt")
    print(f"Device: {device}; model: {config.model}; output: {run_dir}", flush=True)
    history = []
    with (run_dir / "train_res.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["epoch", "loss", "accuracy_percent", "online_loss"])
        writer.writeheader()
        for epoch in range(config.epochs):
            set_epoch(model, config, epoch)
            online = run_epoch(loader, model, device, config, optimizer)
            # Preserve fractional delays and the training-time smoothing for measurement.
            final = run_epoch(measure_loader, model, device, config)
            row = {"epoch": epoch + 1, "loss": final["loss"],
                   "accuracy_percent": final["accuracy_percent"], "online_loss": online["loss"]}
            history.append(row)
            writer.writerow(row)
            stream.flush()
            scheduler.step()
            if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == config.epochs:
                print(f"Epoch {epoch + 1}/{config.epochs}: loss={final['loss']:.6f}, "
                      f"training accuracy={final['accuracy_percent']:.2f}%", flush=True)
    final.update(epoch=config.epochs, model=config.model, seed=config.seed,
                 dataset_seed=config.dataset_seed, task_type=config.task_type,
                 trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad))
    (run_dir / "final_train.json").write_text(json.dumps(final, indent=2))
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "epoch": config.epochs,
                "config": settings, "metrics": final,
                "recurrent_sigmas": {name: m.sigma for name, m in model.named_modules()
                                     if isinstance(m, axonal_recdel)}}, run_dir / "last.pth")
    plot_results(history, final, config, run_dir)
    print(f"Final training accuracy: {final['accuracy_percent']:.2f}%\nSaved: {run_dir}", flush=True)

    return history, final, run_dir

## Match the six models

Within each pathway, axonal and synaptic models start with equal outputs (checked
on a probe). Hybrid models share the learned base delays and add fixed random
synaptic offsets. Projection weights and biases are also matched across pathways.
Recurrent models have recurrence in every hidden layer and no feedforward delays;
feedforward models have no recurrent connections.

In [9]:
def matched_models(config, pathway="recurrent"):
    """Match weights and initial functions; synaptic delays subsequently untie."""
    suffixes = {'recurrent': 'recurrent_only_delays',
                'feedforward': 'feedforward_only_delays'}
    if pathway not in suffixes:
        raise ValueError('pathway must be recurrent or feedforward')
    suffix = suffixes[pathway]
    config = deepcopy(config)
    if pathway == 'recurrent':
        # Existing recurrent classes honor this flag; the comparison needs
        # recurrence in every hidden layer, including a single-hidden-layer run.
        config.no_recurrence_in_last_layer = False
    ax_config, sy_config = deepcopy(config), deepcopy(config)
    ax_config.delay_pathway = sy_config.delay_pathway = pathway
    ax_config.model = f'SNN_axonal_{suffix}'
    sy_config.model = f'SNN_synaptic_{suffix}'
    seed_everything(config.seed)
    ax = getattr(networks, ax_config.model)(ax_config)
    sy = getattr(networks, sy_config.model)(sy_config)
    ax_weights = [m for m in ax.layers if isinstance(m, torch.nn.Linear)]
    sy_weights = [m for m in sy.layers if isinstance(m, (torch.nn.Linear, dcls_module))]
    ax_delays = [m for m in ax.layers if isinstance(m, dcls_module)]
    sy_delays = [m for m in sy.layers if isinstance(m, dcls_module)]
    ax_recs = [m for m in ax.layers if isinstance(m, axonal_recdel)]
    sy_recs = [m for m in sy.layers if isinstance(m, axonal_recdel)]
    with torch.no_grad():
        for a, s in zip(ax_weights, sy_weights, strict=True):
            s.weight.copy_(a.weight.unsqueeze(-1) if isinstance(s, dcls_module) else a.weight)
            if s.bias is not None:
                s.bias.copy_(a.bias)
        for a, s in zip(ax_delays, sy_delays, strict=True):
            # P axes are (spatial dimension, output channel, input channel, tap).
            s.P.copy_(a.P[:, :, 0, :].unsqueeze(1).expand_as(s.P))
        for a, s in zip(ax_recs, sy_recs, strict=True):
            s.recurrent_weights.copy_(a.recurrent_weights)
            s.recurrent_delays.copy_(a.recurrent_delays[None, :].expand_as(s.recurrent_delays))
            if a.use_rec_bias:
                s.recurrent_bias.copy_(a.recurrent_bias)
        for model, cfg in ((ax, ax_config), (sy, sy_config)):
            set_epoch(model, cfg, 0)
            model.eval()
        generator = torch.Generator().manual_seed(123)
        probe = torch.rand(config.time_window, 2, config.input_size, generator=generator)
        torch.testing.assert_close(ax(probe), sy(probe), atol=1e-5, rtol=1e-5)
        reset_states(ax)
        reset_states(sy)
    hy_config = deepcopy(config)
    hy_config.delay_pathway = pathway
    hy_config.model = f'SNN_hybrid_{suffix}'
    hy = getattr(networks, hy_config.model)(hy_config)
    # Copy common weights/biases and the learned base delays. Fixed offsets are
    # retained: hybrid starts with different effective delays by design.
    with torch.no_grad():
        for source, target in zip(sy.layers, hy.layers, strict=True):
            if isinstance(target, dcls_module):
                target.weight.copy_(source.weight)
                if target.bias is not None:
                    target.bias.copy_(source.bias)
                learned_delay_parameter(target, 'P').copy_(source.P[:, :1])
            elif isinstance(target, axonal_recdel):
                target.recurrent_weights.copy_(source.recurrent_weights)
                if target.use_rec_bias:
                    target.recurrent_bias.copy_(source.recurrent_bias)
                learned_delay_parameter(target, 'recurrent_delays').copy_(source.recurrent_delays[0])
                if hasattr(target, 'p_spread'):
                    target.p_spread.copy_(source.p_spread)
            else:
                target.load_state_dict(source.state_dict())
    return [(ax_config, ax), (sy_config, sy), (hy_config, hy)]


def six_models(config):
    """Delay location × delay type; match connection weights across locations."""
    recurrent = matched_models(config, pathway='recurrent')
    feedforward = matched_models(config, pathway='feedforward')
    models = []
    for kind, (rc, rec), (fc, ff) in zip(
            ('Axonal', 'Synaptic', 'Hybrid'), recurrent, feedforward, strict=True):
        # All projections have the same connectivity, but their delay operators
        # differ. Match connection weights/biases without copying delay tensors.
        rec_projections = [m for m in rec.layers if isinstance(m, torch.nn.Linear)]
        ff_projections = [m for m in ff.layers if isinstance(m, torch.nn.Linear)
                          or (isinstance(m, dcls_module) and m.weight.requires_grad)]
        with torch.no_grad():
            for source, target in zip(rec_projections, ff_projections, strict=True):
                target.weight.copy_(source.weight.unsqueeze(-1)
                                    if isinstance(target, dcls_module) else source.weight)
                if target.bias is not None:
                    target.bias.copy_(source.bias)
        assert not any(isinstance(m, dcls_module) for m in rec.modules())
        assert not any(isinstance(m, axonal_recdel) for m in ff.modules())
        models.extend([(f'Recurrent {kind}', rc, rec),
                       (f'Feedforward {kind}', fc, ff)])
    return models

## Run comparison

Run this cell again after editing and rerunning Configuration. It creates fresh models,
checks dataset equality, and saves the same per-model CSV, JSON, dataset, checkpoint
and summary plots as the script. Models train sequentially on the selected device.

In [ ]:
if DEVICE not in ('auto', 'cpu', 'cuda'):
    raise ValueError('DEVICE must be auto, cpu, or cuda')
if min(config.epochs, config.num_samples, config.batch_size, config.cpu_threads,
       config.input_size, config.output_size, config.time_window, *config.hidden_layers) < 1:
    raise ValueError('Epochs, samples, batch size, CPU threads and dimensions must be positive')
if config.task_type not in ('temporal', 'spatial') or config.readout not in ('mean', 'sum', 'last'):
    raise ValueError('Invalid task_type or readout')
device = torch.device(('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE)
if device.type == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA requested but unavailable; enable a GPU accelerator or use DEVICE="auto"')
torch.set_num_threads(config.cpu_threads)
# Keep the settings associated with these results even if Configuration is rerun later.
run_config = deepcopy(config)
out = OUTPUT_ROOT / f'{config.task_type}_seed{config.seed}_{datetime.now():%Y-%m-%d-%H-%M-%S-%f}'
models = six_models(run_config)
histories, results = {}, {}
reference_data = None
for label, cfg, model in models:
    directory = out / label.lower().replace(' ', '_')
    history, final, _ = run(cfg, device, directory, model=model)
    final['feedforward_delay_parameters'] = sum(learned_delay_parameter(m, 'P').numel() for m in model.layers if isinstance(m, dcls_module))
    final['recurrent_delay_parameters'] = sum(learned_delay_parameter(m, 'recurrent_delays').numel() for m in model.layers if isinstance(m, axonal_recdel))
    final['fixed_synaptic_offsets'] = sum(b.numel() for n, b in model.named_buffers() if n.endswith('.offsets'))
    results[label], histories[label] = final, history
    data = torch.load(directory / 'dataset.pt', weights_only=True)
    if reference_data is None:
        reference_data = data
    else:
        assert all(torch.equal(data[k], reference_data[k]) for k in data)
    # Free device memory before training the next model.
    model.cpu()

(out / 'comparison.json').write_text(json.dumps({
    'initialization': 'Matched connection weights and biases across delay locations. Recurrent models have only linear FF projections; feedforward models have no recurrent connections. Within each location, delay initialization is matched across types, with added fixed offsets for hybrid. Same dataset and batch order for all six models.',
    'results': results,
}, indent=2))
print(f'Six-model comparison saved: {out}', flush=True)

Device: cpu; model: SNN_axonal_recurrent_only_delays; output: /Users/mattia/Documents/CentraleSupelec/3A/filiereRecherche/EtudeDeCas/Code/DelRec/notebooks/exp/MEM/delay_location_comparison/temporal_seed0_2026-09-13-12-24-00-120826/recurrent_axonal
Epoch 1/200: loss=2.772580, training accuracy=6.64%
Epoch 10/200: loss=2.771578, training accuracy=9.38%
Epoch 20/200: loss=2.739926, training accuracy=10.16%
Epoch 30/200: loss=2.708595, training accuracy=9.77%
Epoch 40/200: loss=2.591131, training accuracy=12.50%
Epoch 50/200: loss=2.538614, training accuracy=16.41%
Epoch 60/200: loss=2.470519, training accuracy=19.14%
Epoch 70/200: loss=2.318409, training accuracy=25.00%
Epoch 80/200: loss=2.242057, training accuracy=25.39%
Epoch 90/200: loss=2.151452, training accuracy=26.56%
Epoch 100/200: loss=2.197585, training accuracy=30.47%
Epoch 110/200: loss=2.105754, training accuracy=30.86%
Epoch 120/200: loss=2.123342, training accuracy=35.94%
Epoch 130/200: loss=2.066753, training accuracy=37.

## Plot comparison

This cell can be rerun without training. Solid curves show recurrent delays;
dashed curves and hatched bars show feedforward delays. The PNG/PDF comparison,
`comparison.json` and per-model folders are saved under the printed output path
(in `/kaggle/working/exp/MEM/delay_location_comparison/` on Kaggle).

In [ ]:
def plot_comparison(histories, results, config, out):
    fig = plt.figure(figsize=(16, 9), layout='constrained')
    grid = fig.add_gridspec(2, 2)
    loss_ax, acc_ax, bar_ax = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, :])
    colors = {'Axonal': 'tab:blue', 'Synaptic': 'tab:orange', 'Hybrid': 'tab:green'}
    for label, history in histories.items():
        family, kind = label.split()
        style = '-' if family == 'Recurrent' else '--'
        epochs = [r['epoch'] for r in history]
        loss_ax.plot(epochs, [r['loss'] for r in history], color=colors[kind], linestyle=style, label=label)
        acc_ax.plot(epochs, [r['accuracy_percent'] for r in history], color=colors[kind], linestyle=style, label=label)
    loss_ax.set(xlabel='Epoch', ylabel='Cross-entropy loss', title='Training loss')
    acc_ax.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Training accuracy', ylim=(0, 105))
    for axis in (loss_ax, acc_ax):
        axis.legend(fontsize=9, ncol=2)
        axis.grid(alpha=0.25)
    labels = [f"{label.replace(' ', chr(10), 1)}\n{r['trainable_parameters']:,} parameters" for label, r in results.items()]
    bars = bar_ax.bar(labels, [r['accuracy_percent'] for r in results.values()],
                      color=[colors[label.split()[1]] for label in results])
    for bar, label in zip(bars, results):
        if label.startswith('Feedforward '):
            bar.set_hatch('//')
    bar_ax.bar_label(bars, labels=[f"{r['accuracy_percent']:.2f}%" for r in results.values()], padding=4)
    bar_ax.set(ylabel='Accuracy (%)', title='Final training accuracy', ylim=(0, 110))
    topology = ' → '.join(map(str, [config.input_size] + config.hidden_layers + [config.output_size]))
    fig.suptitle(f'Recurrent-only vs feedforward-only delays | {config.task_type}, {config.num_samples} samples, topology {topology}\n'
                 'Solid: recurrent delays only · Dashed: feedforward delays only')
    for extension in ('png', 'pdf'):
        fig.savefig(out / f'delay_location_comparison.{extension}', dpi=180)
    plt.show()
    plt.close(fig)


plot_comparison(histories, results, run_config, out)